<a href="https://colab.research.google.com/github/diwakarasd/Test/blob/main/AIHelperHub_Keyword_NLP_mapping_with_URLs_in_sitemap_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install trafilatura

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import trafilatura
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [10]:
url_counter = 1;
def parse_sitemap(sitemap_url):
    """
    Parses an XML sitemap file and returns a list of non-image URLs.
    """

    try:
        response = requests.get(sitemap_url)
        response.raise_for_status()  # Raise an exception for non-200 status codes
        print(f"inside function ")
        # Check for successful response
        if response.status_code == 200:
            global url_counter;
            soup = BeautifulSoup(response.content, 'xml')
            urls = []
            image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.svg')

            for url_tag in soup.find_all('loc'):
                url = url_tag.text.strip()
                if not url.lower().endswith(image_extensions):  # Check for image file extensions
                    #if "/search/" in url and "?" not in url:
                      urls.append(url)

            for url in urls:
                print(f"URL {url_counter} found: {url}")
                url_counter +=1

            return urls
        else:
            print(f"Error retrieving sitemap: Status code {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error fetching sitemap from {sitemap_url}: {e}")
        return None

# Replace 'https://www.example.com/sitemap.xml' with the actual sitemap URL
sitemap_url1 = "https://aihelperhub.com/post-sitemap.xml"


# Call the parse_sitemap function to retrieve URLs from the sitemap
urls = parse_sitemap(sitemap_url1)



inside function 
URL 1 found: https://aihelperhub.com/blog/python-seo/automating-keyword-research-with-n-gram-analysis/
URL 2 found: https://aihelperhub.com/blog/advanced-seo/ultimate-guide-to-seo-technical-audit-checklist/
URL 3 found: https://aihelperhub.com/blog/advanced-seo/best-seo-content-optimization-tools/
URL 4 found: https://aihelperhub.com/blog/python-seo/google-colab-for-seo/
URL 5 found: https://aihelperhub.com/blog/advanced-seo/content-optimization-guide/
URL 6 found: https://aihelperhub.com/blog/python-seo/how-to-use-python-for-seo/
URL 7 found: https://aihelperhub.com/blog/advanced-seo/how-to-create-topical-map/
URL 8 found: https://aihelperhub.com/blog/python-seo/nlp-content-clustering-for-1000-urls-in-python/
URL 9 found: https://aihelperhub.com/blog/advanced-seo/how-to-utilize-nlp-to-optimize-your-seo/
URL 10 found: https://aihelperhub.com/blog/generative-ai/separate-vocals-from-any-song-with-rvc-in-google-colab-make-your-own-karaoke-version/
URL 11 found: https://ai

In [11]:
## Create a requests session with a larger connection pool
session = requests.Session()
adapter = HTTPAdapter(pool_connections=100, pool_maxsize=100)
session.mount('http://', adapter)
session.mount('https://', adapter)
counter = 1;
def fetch_text_from_url(session, url):
    print(f"Processing URL: {url}")  # Print the URL being processed
    global counter;
    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            text = trafilatura.extract(downloaded)
            if text:
                print(f"URL No {counter} Successfully fetched text from {url}")  # Print on successful fetch
                counter += 1
                return text
            else:
                print(f"Failed to extract text from {url}.")
                return None
        else:
            print(f"Failed to download content from {url}.")
            return None
    except Exception as e:
        print(f"Error fetching text from URL: {e}")
        return None

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = ''.join([char for char in text if char.isalpha() or char.isspace()])
    # Tokenize text
    words = word_tokenize(text)
    # Remove stopwords
    stop_words_english = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words_english]
    # Rejoin words into a single string
    text = ' '.join(words)
    return text

def extract_texts(urls):
    documents = []

    # Use ThreadPoolExecutor to fetch texts in parallel
    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_url = {executor.submit(fetch_text_from_url, session, url): url for url in urls}
        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                text = future.result()
                if text:
                    print(f"Processing text for {url}")  # Print before processing text
                    processed_text = preprocess_text(text)
                    documents.append((url, processed_text))
                else:
                    print(f"Failed to retrieve text from {url}. Skipping.")
            except Exception as e:
                print(f"Error processing {url}: {e}")

    if not documents:
        print("No text retrieved from any URL. Exiting.")
        return None

    return documents


if urls:
    documents = extract_texts(urls)
else:
    print("Failed to retrieve URLs from the sitemap.")

Processing URL: https://aihelperhub.com/blog/python-seo/automating-keyword-research-with-n-gram-analysis/Processing URL: https://aihelperhub.com/blog/advanced-seo/ultimate-guide-to-seo-technical-audit-checklist/

Processing URL: https://aihelperhub.com/blog/advanced-seo/best-seo-content-optimization-tools/
Processing URL: https://aihelperhub.com/blog/python-seo/google-colab-for-seo/
Processing URL: https://aihelperhub.com/blog/advanced-seo/content-optimization-guide/
Processing URL: https://aihelperhub.com/blog/python-seo/how-to-use-python-for-seo/
Processing URL: https://aihelperhub.com/blog/advanced-seo/how-to-create-topical-map/
Processing URL: https://aihelperhub.com/blog/python-seo/nlp-content-clustering-for-1000-urls-in-python/
Processing URL: https://aihelperhub.com/blog/advanced-seo/how-to-utilize-nlp-to-optimize-your-seo/
Processing URL: https://aihelperhub.com/blog/generative-ai/separate-vocals-from-any-song-with-rvc-in-google-colab-make-your-own-karaoke-version/
URL No 1 Suc

URL No 4 Successfully fetched text from https://aihelperhub.com/blog/advanced-seo/how-to-create-topical-map/
Processing URL: https://aihelperhub.com/blog/advanced-seo/how-to-implement-eeat-principles-to-improve-seo/
Processing text for https://aihelperhub.com/blog/advanced-seo/how-to-create-topical-map/
URL No 5 Successfully fetched text from https://aihelperhub.com/blog/generative-ai/separate-vocals-from-any-song-with-rvc-in-google-colab-make-your-own-karaoke-version/
Processing URL: https://aihelperhub.com/blog/generative-ai/how-to-use-google-nlp-tool-for-seo/
Processing text for https://aihelperhub.com/blog/generative-ai/separate-vocals-from-any-song-with-rvc-in-google-colab-make-your-own-karaoke-version/


URL No 6 Successfully fetched text from https://aihelperhub.com/blog/advanced-seo/ultimate-guide-to-seo-technical-audit-checklist/
Processing URL: https://aihelperhub.com/blog/generative-ai/how-to-find-lsi-keywords-using-custom-gpt/
Processing text for https://aihelperhub.com/blog/advanced-seo/ultimate-guide-to-seo-technical-audit-checklist/
URL No 7 Successfully fetched text from https://aihelperhub.com/blog/python-seo/google-colab-for-seo/
Processing URL: https://aihelperhub.com/blog/generative-ai/how-to-write-a-blog-outline-using-custom-gpt/
Processing text for https://aihelperhub.com/blog/python-seo/google-colab-for-seo/
URL No 8 Successfully fetched text from https://aihelperhub.com/blog/advanced-seo/best-seo-content-optimization-tools/
Processing URL: https://aihelperhub.com/blog/generative-ai/how-to-create-a-custom-gpt-for-seo/
Processing text for https://aihelperhub.com/blog/advanced-seo/best-seo-content-optimization-tools/
URL No 9 Successfully fetched text from https://aihelp

URL No 20 Successfully fetched text from https://aihelperhub.com/blog/generative-ai/top-10-custom-gpts-for-seo/
Processing URL: https://aihelperhub.com/blog/advanced-seo/what-is-topical-authority/
Processing text for https://aihelperhub.com/blog/generative-ai/top-10-custom-gpts-for-seo/
URL No 21 Successfully fetched text from https://aihelperhub.com/blog/generative-ai/how-to-use-chatgpt-for-seo-keyword-research/
Processing text for https://aihelperhub.com/blog/generative-ai/how-to-use-chatgpt-for-seo-keyword-research/


URL No 22 Successfully fetched text from https://aihelperhub.com/blog/python-seo/how-use-google-indexing-api-with-python/
Processing text for https://aihelperhub.com/blog/python-seo/how-use-google-indexing-api-with-python/
URL No 23 Successfully fetched text from https://aihelperhub.com/blog/python-seo/how-to-find-contextual-internal-links-on-website/
Processing text for https://aihelperhub.com/blog/python-seo/how-to-find-contextual-internal-links-on-website/


URL No 24 Successfully fetched text from https://aihelperhub.com/blog/python-seo/how-to-check-existing-backlinks-and-their-quality-dofollow-or-nofollow/
Processing text for https://aihelperhub.com/blog/python-seo/how-to-check-existing-backlinks-and-their-quality-dofollow-or-nofollow/


URL No 25 Successfully fetched text from https://aihelperhub.com/blog/python-seo/how-to-find-existing-pillar-pages-and-their-cluster-topics-of-any-website/
Processing text for https://aihelperhub.com/blog/python-seo/how-to-find-existing-pillar-pages-and-their-cluster-topics-of-any-website/


URL No 26 Successfully fetched text from https://aihelperhub.com/blog/python-seo/how-to-crawl-a-website-using-python/
Processing text for https://aihelperhub.com/blog/python-seo/how-to-crawl-a-website-using-python/


URL No 27 Successfully fetched text from https://aihelperhub.com/blog/generative-ai/custom-gpts-for-seo/
Processing text for https://aihelperhub.com/blog/generative-ai/custom-gpts-for-seo/
URL No 28 Successfully fetched text from https://aihelperhub.com/blog/generative-ai/how-to-write-perfect-emails-using-chatgpt/
Processing text for https://aihelperhub.com/blog/generative-ai/how-to-write-perfect-emails-using-chatgpt/


URL No 29 Successfully fetched text from https://aihelperhub.com/blog/python-seo/semantic-keyword-clustering-python/
Processing text for https://aihelperhub.com/blog/python-seo/semantic-keyword-clustering-python/
URL No 30 Successfully fetched text from https://aihelperhub.com/blog/advanced-seo/what-is-topical-authority/
Processing text for https://aihelperhub.com/blog/advanced-seo/what-is-topical-authority/


In [20]:
def vectorize_documents(documents):
    """
    Vectorizes the documents using CountVectorizer.
    Returns URLs and vectorized representations of texts.
    """
    urls, texts = zip(*documents)  # Unzip the tuples into separate lists
    # Vectorize the documents
    print("Vectorizing documents...")  # Print when starting vectorization
    vectorizer = CountVectorizer(stop_words='english')  # Adjust parameters as needed
    text_vectorized = vectorizer.fit_transform(texts)
    return urls, text_vectorized, vectorizer

def vectorize_keyword(keyword, vectorizer):
    """
    Vectorizes the given keyword using the same vectorizer as the documents.
    """
    return vectorizer.transform([keyword])

def group_urls_by_keyword(urls, text_vectorized, vectorizer, keyword, similarity_threshold):
    """
    Groups URLs based on cosine similarity with a given keyword.
    Prints URLs that have a similarity score above the threshold.
    """
    print("Vectorizing keyword...")  # Print when vectorizing keyword
    keyword_vectorized = vectorize_keyword(keyword, vectorizer)

    print("Calculating similarity scores...")  # Print when calculating similarity
    similarities = cosine_similarity(text_vectorized, keyword_vectorized).flatten()

    # Filter URLs that have similarity above the threshold
    matched_urls = [urls[i] for i in range(len(urls)) if similarities[i] >= similarity_threshold]

    # Print the grouped URLs
    if matched_urls:
        print("\nURLs matching the given keyword based on cosine similarity:")
        for url in matched_urls:
            print(f" - {url}")
    else:
        print("\nNo URLs matched the given keyword above the threshold.")


if documents:
    urls, text_vectorized, vectorizer = vectorize_documents(documents)
    similarity_threshold = 0.1  # Adjust the similarity threshold here (e.g., 70%)
    group_urls_by_keyword(urls, text_vectorized, vectorizer,"topical authority", similarity_threshold)
else:
    print("Failed to retrieve URLs from the sitemap.")

Vectorizing documents...
Vectorizing keyword...
Calculating similarity scores...

URLs matching the given keyword based on cosine similarity:
 - https://aihelperhub.com/blog/advanced-seo/how-to-create-topical-map/
 - https://aihelperhub.com/blog/advanced-seo/topical-maps-in-seo/
 - https://aihelperhub.com/blog/advanced-seo/what-is-topical-authority/
